In [ ]:
import pandas as pd
%pip install altair vega_datasets
import altair as alt
alt.renderers.enable()

*Data Cleaning and Preview*

In [ ]:
#gene daata
data=pd.read_csv("datasets\TCGA_BRCA_tpm.tsv", sep="\t")

#clinic data
clinic= pd.read_csv("datasets\\brca_tcga_pan_can_atlas_2018_clinical_data_filtered.tsv", sep="\t")


Preview of genetic sequnce data

In [ ]:
data.head()

Preview of Clinic data

In [ ]:
clinic.head()

Counts of Samples per Subtype

In [ ]:
clinic_summary=clinic.groupby("Subtype").size().rename("Count").reset_index()
clinic_summary

In [ ]:
alt.Chart(clinic_summary).mark_bar().encode(
    alt.X("Subtype"),
    alt.Y("Count"),
    color="Subtype",
    tooltip=["Subtype","Count"]
).properties(
    title="Number of Samples per Subtype",
    width=150,
    height=300
)

*Transposing Genetic Sequencing Data*

In [ ]:
transposed= data.set_index("Ensembl_ID").T.reset_index().rename_axis(None,axis=1).rename(columns={"index":"Ensembl_ID"})
transposed.head()

*Cleaning Up Clinic Data*

In [ ]:
clinic=clinic.rename(columns={"Sample ID":"Ensembl_ID"})
clinic.Ensembl_ID+="A"
clinic.head()

Merging Clinic data with Gene Expression Data

In [ ]:
final_data=pd.merge(transposed,clinic,how='inner',on='Ensembl_ID')
final_data.head()

Keeping Only The subtype and The Gene Expression Data

In [ ]:
df= final_data.drop(columns=['Ensembl_ID','Patient ID',"Diagnosis Age","Cancer Type","Sex","Tumor Type"]).set_index("Subtype").reset_index().dropna()
df.head()

Subtype Samples Counts of the Final Cleaned Dataset

In [ ]:
df.groupby("Subtype").size().reset_index().rename(columns={0:"Count"})

In [ ]:
to_plot = df[["Subtype", "ENSG00000242268.2", "ENSG00000270112.3"]]
to_plot.columns= ['Subtype', 'gene_one', 'gene_two']

alt.Chart(to_plot).mark_circle().encode(
    alt.X("gene_one", scale= alt.Scale(zero=False)),
    alt.Y("gene_two", scale= alt.Scale(zero=False)), 
    color= "Subtype",
    size= "gene_one"
).facet(
    facet="Subtype",
    columns=2
)

# Machine Learning Model 

In [ ]:
X = df.drop("Subtype", axis=1)
y = df.Subtype

Splitting Data into training and testing sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.metrics import accuracy_score

max_depth_best = []
for i in range(1,20):
    clf = rf(max_depth = i, random_state = 42)
    clf.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    max_depth_best.append([i,accuracy_score(y_test,y_pred)])
max_depth = pd.DataFrame(max_depth_best,columns= ["max_depth", "Accuracy"])
max_depth

In [ ]:
max_depth_chart = alt.Chart(max_depth).mark_line().encode(
    alt.X("max_depth", scale= alt.Scale(zero=False)),
    alt.Y("Accuracy", scale= alt.Scale(zero=False)),
    tooltip= ["max_depth","Accuracy"]
).properties(title="Random Forest Accuracy vs Max Depth")

max_depth_chart

In the output above, we see that in range the best level of depth of trees for output is 11,12,13. so we will take max_depth as 12

Choosing the Optiomal n_estimator from range (10,200)

In [ ]:
#Find best n_estimator
n_estimators=[]
for i in range(10,200):
    clf = rf(max_depth=10, n_estimators=i, random_state=42)
    clf.fit(X_train,y_train)
    y_pred= clf.predict(X_test)
    n_estimators.append([i,accuracy_score(y_test,y_pred)])

In [ ]:
n_estimate = pd.DataFrame(n_estimators, columns=["n_estimator", "Accuracy"])
estimate_chart = alt.Chart(n_estimate).mark_line().encode(
    alt.X("n_estimator", scale=alt.Scale(zero=False)),
    alt.Y("Accuracy", scale=alt.Scale(zero=False)),
    tooltip=["n_estimator", "Accuracy"]
).properties(title="Random Forest Accuracy vs n_estimator")

max_depth_chart|estimate_chart

In [ ]:
#The Optimal n_estimator
n_estimate.loc[n_estimate["Accuracy"].idxmax()]

Training the random Forest Model With The Hyperparameters Obtained Above

In [ ]:
rf_model = rf(max_depth=12, n_estimators= 169, random_state=42)
rf_model.fit(X_train, y_train)

#Predicting on Testing Set

y_pred= rf_model.predict(X_test)
accuracy_score(y_test, y_pred)

Random Forest Classifier Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

conf_mat = confusion_matrix(y_test, y_pred, labels= rf_model.classes_)
ConfusionMatrixDisplay(conf_mat, display_labels=rf_model.classes_).plot()

__Random Forest Observations__\
The model is having trouble with predicting BRCA_Normal. This makes sence given that we have very few samples of this subtype.\
Also, The model is having problems with sepparating between BRCA_LumA and BRCA_LumB

__feature impoortance__\
Having built the Random Forest Model, we will now look at the most important genes in the classification of our gene subtypes. Since there are many genes in the model, we will only look at the 10 most important ones.

In [ ]:
sorted_idx = (-rf_model.feature_importances_).argsort()[:10]
feature_data = pd.DataFrame({"Feature" : X.columns[sorted_idx], "Importance":  rf_model.feature_importances_[sorted_idx] }).sort_values(by="Importance", ascending= False)
feature_data["Feature Names"]= [ "DRAIC", "TTC6", "SLC7A13", "DNAI7", "CCNB2", "CENPA", "ESR1", "RAB6C", "CDC20",  "CCNE1"]
feature_data

In [ ]:
alt.Chart(feature_data).mark_bar().encode(
    y= alt.Y("Feature Names", sort=alt.EncodingSortField(field="Importance", op="count", order='ascending'), title="Gene"),
    x=alt.X('Importance:Q'),
    color= alt.Color("Feature Names", title= "Gene", legend= None),
    tooltip = ["Feature Names", "Importance"]
)

## Support Vector Machines(SVM) Model

__Finding The Optimal C parameter__

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

svm_class = []
for i in range(1,10):
    clf = make_pipeline(StandardScaler(), SVC(C = i))
    clf.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    svm_class.append([i,accuracy_score(y_test,y_pred)])

svm_class = pd.DataFrame(svm_class, columns=['C parameter', "Accuracy"])
svm_class.head()

In [ ]:
alt.Chart(svm_class).mark_line().encode(
    alt.X("C parameter", scale= alt.Scale(zero=False)),
    alt.Y("Accuracy", scale= alt.Scale(zero=False))
).properties(title="ACCURACY VS C Parameter SVM")

Here, we see that the best C is 2

__Training SVM and Obtaining Confusion Matrix__

In [ ]:
svm_model = make_pipeline(StandardScaler(), SVC(C = 2))

#Fitting model
svm_model.fit(X_train, y_train)

#Obtaining prediction
y_pred_svm = svm_model.predict(X_test)

#Getting Confusion Matrix
svm_conf_mat = confusion_matrix(y_test, y_pred_svm, labels= svm_model.classes_)
ConfusionMatrixDisplay(svm_conf_mat, display_labels=svm_model.classes_).plot()

__SVM Observation__

The matrix shows that this model is not better than random forest in any subclassification